# 03. 인수인계 기본 구조 둘러보기
PR #29의 첫 번째 묶음입니다. 페르소나, 주장·근거·버전 계약, 두 Agent 기본 계산, 품질 검사, Verifier를 순서대로 봅니다.

DB·API 키 없이 실행됩니다. 위에서부터 차례로 실행하세요(메뉴 Run → Run All Cells).
예시 자료는 합성·모의 데이터입니다. 실제 기업 분석 결과가 아닙니다.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'agent/neon_agents.py').exists()), None)
if ROOT is None:
    raise RuntimeError('압축 전체를 해제한 프로젝트/notebooks에서 실행하세요.')
sys.path.insert(0, str(ROOT))
pd.set_option('display.max_colwidth', 120)
print('프로젝트 위치 확인 완료')

프로젝트 위치 확인 완료


## 1. 페르소나: 금융 이해도 4단계 × 투자 성향 2가지 = 8개 조합
L1 입문 · L2 기초 · L3 중급 · L4 고급 / A 적극형 · C 신중형. 미확정 성향은 U(중립)로 처리합니다.
페르소나는 설명 방식만 바꾸고 수치·근거·보류 기준은 바꾸지 않습니다.

In [2]:
from agent.personas import assess_questionnaire, confirm_persona, get_persona_catalog, get_questionnaire

catalog = get_persona_catalog()
display(pd.DataFrame(catalog['literacy'])[['code', 'name', 'definition']])
display(pd.DataFrame(catalog['investment_style'])[['code', 'name', 'definition']])
display(pd.DataFrame(catalog['combined'])[['persona_id', 'name', 'instruction']])

,code,name,definition
0,L1,입문,금융 용어와 정보의 기본 구조를 배우는 사용자
1,L2,기초,주요 금융 개념과 개별 지표의 의미를 이해하는 사용자
2,L3,중급,여러 자료·지표를 비교하고 관계를 검토하는 사용자
3,L4,고급,분석의 가정·산식·자료 편향·한계까지 검토하는 사용자


,code,name,definition
0,A,적극형,기회가 성립하는 조건을 우선 검토하는 분석 관점
1,C,신중형,지속 가능성·불확실성·위험을 우선 검토하는 분석 관점


,persona_id,name,instruction
0,L1-A,입문·적극형,"쉬운 말과 비유로 핵심을 먼저 설명하고, 처음 나오는 용어를 풀이한다.기회와 성립 조건을 먼저 제시하되 실패 조건·반대 근거·위험을 함께 제시한다."
1,L1-C,입문·신중형,"쉬운 말과 비유로 핵심을 먼저 설명하고, 처음 나오는 용어를 풀이한다.위험과 하방 조건을 먼저 제시하되 기회 요인·위험 완화 근거도 함께 제시한다."
2,L2-A,기초·적극형,"지표의 의미, 해당 기업의 사례, 해석 시 주의점을 차례로 연결한다.기회와 성립 조건을 먼저 제시하되 실패 조건·반대 근거·위험을 함께 제시한다."
3,L2-C,기초·신중형,"지표의 의미, 해당 기업의 사례, 해석 시 주의점을 차례로 연결한다.위험과 하방 조건을 먼저 제시하되 기회 요인·위험 완화 근거도 함께 제시한다."
4,L3-A,중급·적극형,"비교표와 지표 간 관계, 반대 해석, 추가 확인 사항을 제시한다.기회와 성립 조건을 먼저 제시하되 실패 조건·반대 근거·위험을 함께 제시한다."
5,L3-C,중급·신중형,"비교표와 지표 간 관계, 반대 해석, 추가 확인 사항을 제시한다.위험과 하방 조건을 먼저 제시하되 기회 요인·위험 완화 근거도 함께 제시한다."
6,L4-A,고급·적극형,"원천 자료, 방법론, 계산 조건, 민감도와 반증 조건을 상세히 제시한다.기회와 성립 조건을 먼저 제시하되 실패 조건·반대 근거·위험을 함께 제시한다."
7,L4-C,고급·신중형,"원천 자료, 방법론, 계산 조건, 민감도와 반증 조건을 상세히 제시한다.위험과 하방 조건을 먼저 제시하되 기회 요인·위험 완화 근거도 함께 제시한다."


### 설문으로 페르소나 제안받기
지식 문항 8개(점수 → L등급)와 성향 문항 4개(A/C 다수 → 성향). 결과는 **제안**이며 사용자가 확인해야 확정됩니다.

In [3]:
questionnaire = get_questionnaire()
display(pd.DataFrame(questionnaire['knowledge'])[['id', 'question']])

# 예시 답변(가상의 사용자): 지식 8문항 중 5개 정답, 성향은 A가 3개
knowledge = {'K1': 'b', 'K2': 'c', 'K3': 'a', 'K4': 'c', 'K5': 'a', 'K6': 'a', 'K7': 'a', 'K8': 'a'}
style = {'S1': 'A', 'S2': 'A', 'S3': 'A', 'S4': 'C'}
assessed = assess_questionnaire(knowledge, style)
print('지식 점수:', assessed['knowledge_score'], '/ 성향 응답 수:', assessed['style_counts'])
print('제안 결과:', assessed['profile']['persona_id'], '상태:', assessed['profile']['classification_status'])
print('참고:', assessed['reasons'])

,id,question
0,K1,매출과 이익에 대한 설명 중 맞는 것은?
1,K2,이익률을 비교할 때 먼저 확인할 것은?
2,K3,사업모델 비교군 선정에서 중요한 것은?
3,K4,순이익과 영업현금흐름의 차이에 대한 설명은?
4,K5,이동평균에 대한 설명 중 맞는 것은?
5,K6,RSI 등 지표 하나로 다음 가격을 확정할 수 있는가?
6,K7,액면분할 전후 가격을 비교할 때 중요한 것은?
7,K8,지표 설정을 여러 번 바꿔 가장 좋은 결과만 고르면?


지식 점수: 5 / 성향 응답 수: {'A': 3, 'C': 1}
제안 결과: L3-A 상태: suggested
참고: ['지식 점수 임계값 0~2/3~4/5~6/7~8은 검증 전 초기 규칙입니다.']


In [4]:
from agent.personas import PersonaProfile

suggested = PersonaProfile.from_dict(assessed['profile'])
confirmed = confirm_persona(suggested)  # 사용자가 화면에서 '맞아요'를 눌렀다고 가정
print(suggested.persona_id, suggested.status, '→', confirmed.persona_id, confirmed.status)

L3-A suggested → L3-A user_confirmed


## 2. 주장·근거·버전 계약 (합성 A기업)
주장(claim) → 근거(evidence) → 문서(document) 순서로 연결하고, 전체 내용의 digest(해시)로 버전을 식별합니다.

In [5]:
from agent.research_contract import revise_claim, validate_report
from examples.research_example import build_example

report = build_example()
validate_report(report)  # 구조·연결·해시가 맞지 않으면 오류
print('보고서:', report['reportId'], '버전', report['contentRevision'], '자료 모드', report['dataMode'])
print('digest:', report['digest'][:16], '...')
display(pd.DataFrame(report['claims'])[['claimId', 'claimVersion', 'domain', 'kind', 'text', 'assessment']])
display(pd.DataFrame(report['evidence'])[['evidenceId', 'documentId', 'location', 'verificationStatus']])

보고서: SAMPLE-A-001 버전 1 자료 모드 synthetic
digest: 82de421af51f8682 ...


,claimId,claimVersion,domain,kind,text,assessment
0,C01,1,industry_business_model,interpretation,고부가 제품 비중 증가는 이익 개선 요인이 될 수 있으나 마진·투자비 확인 전에는 보류한다.,withheld
1,C02,1,technical_analysis,fact,마지막 종가 97는 3거래일 단순이동평균 99.0보다 낮다.,provisional


,evidenceId,documentId,location,verificationStatus
0,E01,D01,"{'type': 'markdown_heading', 'heading': '제품 구성'}",unverified
1,E02,D01,"{'type': 'markdown_heading', 'heading': '비용과 경쟁'}",unverified
2,E03,D02,"{'type': 'json_pointer', 'pointer': '/observations'}",unverified


In [6]:
revised = revise_claim(report, 'C01', text='제품 비중 변화는 확인 대상이나 이익 개선은 미확인이다.',
                       reason='근거 범위를 넘는 표현 수정')
print('수정 전 버전', report['contentRevision'], '→ 수정 후', revised['contentRevision'])
print('digest 변경:', report['digest'][:12], '→', revised['digest'][:12])
print('수정된 주장 상태:', next(c for c in revised['claims'] if c['claimId'] == 'C01')['assessment'])

수정 전 버전 1 → 수정 후 2
digest 변경: 82de421af51f → 107042ad9184
수정된 주장 상태: withheld


## 3. 두 Agent 기본 계산
산업 Agent는 원문 근거를 연결하고, 기술적 Agent는 종가·거래량으로 이동평균·수익률·변동성·거래량 변화를 계산합니다.
`DemoLLM`은 실제 LLM이 아니라 오프라인 예시 문구입니다.

In [7]:
from agent.basic_agents import DemoLLM, IndustryBusinessAgent, TechnicalAnalysisAgent, compare_results

industry = IndustryBusinessAgent(DemoLLM()).run(report)
technical = TechnicalAnalysisAgent(DemoLLM()).run(report)
print('산업 Agent 상태:', industry['status'], '/ 기술 Agent 상태:', technical['status'])
display(pd.DataFrame(industry['calculations'])[['calculationId', 'formula', 'result', 'unit']])
display(pd.DataFrame(technical['calculations'])[['calculationId', 'formula', 'result', 'unit']])
print(technical['explanation'])
print(compare_results(industry, technical)['summary'])

산업 Agent 상태: partial / 기술 Agent 상태: partial


,calculationId,formula,result,unit
0,IND_SHARE_PP_E01,current percentage - prior percentage,10.0,percentage_points


,calculationId,formula,result,unit
0,TECH_SMA,mean(last window closes),99.000000,price
1,TECH_RETURN,last close / previous close - 1,-0.020202,ratio
2,TECH_VOLATILITY,population std of available simple returns,0.016388,ratio
3,TECH_VOLUME_CHANGE,last volume / mean(previous window volumes) - 1,0.166667,ratio


[오프라인 데모 문구 · 실제 LLM 미호출] 마지막 종가 97와 3개 관측 이동평균 99을 비교하면 가격이 낮습니다. 마지막 종가가 3개 관측 이동평균 대비 -2.02%로 "이동평균 아래" 상태입니다. (관측 상태 설명이며 예측·매매 신호가 아닙니다.) 직전 대비 수익률 -2.02%는 "작은 변동(하락)" 구간입니다. (관측 상태 설명이며 예측·매매 신호가 아닙니다.) 마지막 거래량이 직전 3개 평균 대비 +16.67%로 "평이한 거래량"입니다. (관측 상태 설명이며 예측·매매 신호가 아닙니다.) 원문과 반대 근거를 검토하세요.
사업 가설과 가격 관찰은 기간·근거가 다릅니다. 투표나 종합점수로 합치지 않습니다.


## 4. 품질 검사와 Verifier
품질: 실행 가능 / 일부 가능 / 보류. 보류이면 계산·LLM·ML을 실행하지 않습니다.
Verifier는 같은 코드로 다시 실행해 결과가 바뀌지 않았는지 비교합니다(원문 사실 검증·최종 승인은 아님).

In [8]:
from agent.basic_agents import AgentSettings
from agent.quality_gate import assess_report, gate_table, run_checked
from agent.result_verifier import verify_result

gates, checks = assess_report(report, reference_date='2026-10-27')
display(pd.DataFrame(gate_table(gates)))

settings = AgentSettings()
result = run_checked(TechnicalAnalysisAgent(settings=settings), report, gates['technical_analysis'])
verdict = verify_result(report, result, settings)
print('Verifier:', verdict['status'])
display(pd.DataFrame(verdict['checks']))

,기업,Agent,상태,이유,기준일,규칙 버전
0,가상 A기업,산업·사업모델,일부 가능,합성 예시: 실데이터 최신성·거래일 검증 대상 아님; 원문이 주장을 지지하는지 검수 대기; 원문이 주장을 지지하는지 검수 대기; 미확보 자료·보류 결론: 제품별 마진; 경쟁사 공급능력 비교,2026-10-27,preflight-2
1,가상 A기업,기술적 분석,일부 가능,합성 예시: 실데이터 최신성·거래일 검증 대상 아님; 전체 OHLC 필드 미확보 시 종가·거래량 분석만 가능,2026-10-27,preflight-2


Verifier: review_required


,검사,상태,설명
0,입력 버전,통과,동일 입력 스냅샷인지 비교
1,status,통과,동일 코드로 재실행한 기본 결과와 비교
2,claims,통과,동일 코드로 재실행한 기본 결과와 비교
3,evidence,통과,동일 코드로 재실행한 기본 결과와 비교
4,calculations,통과,동일 코드로 재실행한 기본 결과와 비교
5,subject,통과,동일 코드로 재실행한 기본 결과와 비교
6,asOf,통과,동일 코드로 재실행한 기본 결과와 비교
7,dataMode,통과,동일 코드로 재실행한 기본 결과와 비교
8,페르소나 불변,통과,표현 설정(L1~L4·A/C/U)과 관계없이 수치·근거·판정이 같아야 함
9,원문 사실성·주장 지지,검수 필요,"DB 원문 진위, 발췌 관련성, 반대 근거는 사람이 확인해야 함"


In [9]:
# 결과를 몰래 바꾸면 Verifier가 failed로 잡습니다.
tampered = json.loads(json.dumps(result))
tampered['calculations'][0]['result'] = 999
print('변조 결과 Verifier:', verify_result(report, tampered, settings)['status'])

변조 결과 Verifier: failed
